# Configuring Your GPT Model

Welcome to the first step in building our own GPT! Before we can write the code for the Transformer architecture, we first need a blueprint. This notebook will guide you through creating a configuration object that holds all the essential design decisions for our model, from its size to its vocabulary.

By the end of this notebook, you will be able to define the architectural "blueprint" for a GPT model using a clean configuration class. You will understand how key hyperparameters like the number of layers, attention heads, and embedding dimensions directly control the model's size, complexity, and resource needs. This blueprint will be the foundation upon which we build all the other components.

In [ ]:
# Setup cell: We'll use dataclasses to create our configuration object
import dataclasses
import math

### The Core Idea: An Architect's Blueprint

Imagine you're building a skyscraper. You wouldn't just show up with steel beams and start welding. You'd start with a detailed architectural blueprint. This blueprint specifies everything: how many floors the building will have, the dimensions of each floor, the number of support columns, and so on. The blueprint isn't the building itself, but it's the essential plan that governs its construction.

In the world of neural networks, a **configuration object** is our architectural blueprint.

Instead of passing a dozen different arguments to our model's constructor, we bundle them all together into a single, clean object. This object holds all the key hyperparameters that define the model's size and shape.

Here are the most important specifications on our GPT blueprint:

*   **`n_layer`**: The number of Transformer blocks to stack on top of each other. This is like the number of floors in our skyscraper. More layers give the model more "depth" to learn complex relationships in the data.
*   **`n_embd`**: The embedding dimension. This is the size of the vector that represents each token. Think of it as the "width" or "square footage" of each floor in our skyscraper. A larger dimension allows each token to carry more information.
*   **`n_head`**: The number of attention heads. In each Transformer block, the model "attends" to the input sequence from multiple perspectives at once. This is like having multiple sets of structural columns on each floor, each analyzing the building's load from a different angle.
*   **`block_size`**: The maximum context length. This is the number of tokens the model can "see" at once when making a prediction. It's the size of the window the model looks through.
*   **`vocab_size`**: The number of unique tokens in our vocabulary. This is the dictionary of all possible words or sub-words the model knows.

By defining these values up front, we create a single source of truth for our model's architecture, making it easy to experiment with different model sizes and to save and load configurations.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """The architectural blueprint for our GPT model."""
    block_size: int = 256     # The maximum sequence length (context window)
    vocab_size: int = 65      # The number of tokens in our vocabulary
    n_layer: int = 6          # The number of Transformer blocks
    n_head: int = 6           # The number of attention heads in each block
    n_embd: int = 384         # The embedding dimension for each token

### Walking Through the Implementation

Let's break down that simple but powerful piece of code.

**`from dataclasses import dataclass`**
We import the `dataclass` decorator from Python's standard library. A decorator is a special function that adds functionality to another function or class.

**`@dataclass`**
This is the decorator in action. By placing it above our class definition, we tell Python to automatically generate special methods for us, like `__init__` (to initialize the object) and `__repr__` (for a nice, clean printout). This lets us define the *data* we want to store without writing all the boilerplate code to manage it.

**`class GPTConfig:`**
We define a regular Python class named `GPTConfig`. This will serve as our container or "blueprint."

**The Attributes**
Inside the class, we define the attributes we discussed earlier.
*   `block_size: int = 256`: We set the type hint to `int` and provide a default value of 256. This means our model will be able to process sequences of up to 256 tokens.
*   `vocab_size: int = 65`: We're using a tiny vocabulary for our initial character-level model (e.g., all letters, numbers, and some punctuation).
*   `n_layer: int = 6`: Our model will have 6 Transformer blocks stacked vertically.
*   `n_head: int = 6`: Each of those 6 blocks will contain a multi-head attention mechanism with 6 heads.
*   `n_embd: int = 384`: Each token will be represented by a vector of 384 numbers.

The beauty of this approach is its clarity. All the core architectural decisions are listed in one place, with clear names and default values.

In [ ]:
# We can now create an instance of our configuration.
# This creates a "blueprint" for a small, simple GPT model.
tiny_gpt_config = GPTConfig()

# The @dataclass decorator gives us a nice, readable representation
print(tiny_gpt_config)

# Let's create another configuration for a larger model, like GPT-2 (124M params)
# by overriding the default values.
gpt2_small_config = GPTConfig(
    n_layer=12,
    n_head=12,
    n_embd=768,
    block_size=1024,
    vocab_size=50257 # The actual vocab size of GPT-2
)

print("\nConfiguration for a GPT-2-like model:")
print(gpt2_small_config)

### Going Deeper: The Curious Case of `vocab_size`

You may have noticed in the original `nanoGPT` source code, the `vocab_size` is set to `50304`, not `50257` (the actual vocabulary size of the GPT-2 model). Why the difference?

```python
# From nanoGPT/model.py
vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
```

This is a subtle but important optimization for hardware performance. Modern GPUs are incredibly powerful, but they achieve their speed through massive parallelism. They work most efficiently when the dimensions of the matrices they are multiplying are multiples of a certain number, often 8, 16, 32, or 64, due to the architecture of their internal processing units (like Tensor Cores on NVIDIA GPUs).

The `vocab_size` directly determines the width of the final output layer's weight matrix in the model. By padding the vocabulary from 50257 up to 50304 (which is `786 * 64`), the matrix multiplication becomes more efficient. The extra "dummy" vocabulary entries are never used, but their presence in the matrix dimensions allows the GPU to process the calculations faster. This is a perfect example of a practical engineering decision that bridges the gap between pure ML theory and high-performance implementation.

In [ ]:
# Let's visualize how the configuration parameters map to the model's structure.
# This ASCII art diagram shows where each hyperparameter fits in.

def draw_model_architecture(config: GPTConfig):
    """Prints an ASCII art diagram of the GPT model based on the config."""

    diagram = f"""
    GPT Model Architecture (based on your config)
    -------------------------------------------------
    Input Tokens (batch, block_size={config.block_size})
           |
           v
    Embedding Lookup (vocab_size={config.vocab_size}) -> (batch, block_size={config.block_size}, n_embd={config.n_embd})
           |
           v
    +---------------------------------+
    |       TRANSFORMER BLOCK 1       |  ┐
    |  - Self-Attention (n_head={config.n_head}) |  |
    |  - Feed-Forward Network         |  |
    +---------------------------------+  |
    |       TRANSFORMER BLOCK 2       |  | n_layer={config.n_layer}
    |              ...              |  |
    +---------------------------------+  |
    |      TRANSFORMER BLOCK {config.n_layer}      |  ┘
    +---------------------------------+
           |
           v
    Final LayerNorm
           |
           v
    Output Projection -> (batch, block_size={config.block_size}, vocab_size={config.vocab_size})
    """
    print(diagram)

# Create a config for a small model and visualize it
my_config = GPTConfig(n_layer=4, n_head=4, n_embd=128, block_size=128)
draw_model_architecture(my_config)

### Summary

In this notebook, we created the foundational blueprint for our GPT model. We moved beyond just theory and defined a concrete, code-based plan for the model's architecture.

**What we built:**
*   A clean, readable `GPTConfig` dataclass that bundles all essential model hyperparameters into a single object.

**Key Takeaways:**
*   **Separation of Concerns:** A configuration object neatly separates the model's *architecture* (the "what") from its *implementation* (the "how").
*   **Hyperparameters Drive Complexity:** The values in `GPTConfig`, especially `n_layer`, `n_head`, and `n_embd`, directly dictate the size, number of parameters, and computational cost of the model. Doubling `n_embd` roughly quadruples the parameters in many key layers.
*   **From Blueprint to Building:** This configuration object is not the model itself, but it is the essential first step. Every component we build from now on will read from this config object to know its own size and shape.

**What's Next:**

Now that we have the architect's blueprint, it's time to lay the foundation and start building. The core of any GPT model is the Transformer Block. In the next notebook, **Building the GPT Transformer Block**, we will implement the first "floor" of our skyscraper, starting with the powerful multi-head self-attention mechanism.